# 2. Create raw-data manifests

Create video and camera-name CSV files from the data staged by notebook 1.

**Rerun behaviour** is controlled by `OVERWRITE`:

- `OVERWRITE = False` (default): existing rows (and their order, which notebook 3 relies on for `VIDEO_LIMIT`) are kept and only entries not yet listed are appended.
- `OVERWRITE = True`: both manifests are rewritten from scratch.

When run from `00_run_annotation_pipeline.ipynb`, the runner's `OVERWRITE` takes precedence.

In [1]:
import csv
import os
from pathlib import Path

# True: rewrite both manifests from scratch.
# False: keep existing rows and append only entries that are not listed yet.
OVERWRITE = False
OVERWRITE = os.environ.get("PIPELINE_OVERWRITE", str(OVERWRITE)).strip().lower() in {"1", "true", "yes"}

print("=" * 72, flush=True)
print(f"Stage 2: creating raw-data manifests (OVERWRITE={OVERWRITE})", flush=True)
print("=" * 72, flush=True)

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebooks").is_dir() and (parent / "annotation_pipeline").is_dir()
)
DATA_DIR = PROJECT_ROOT / "data" / "raw"
clip_list = DATA_DIR / "annotations" / "kitware" / "list-of-annotated-meva-clips.txt"
metadata_dir = DATA_DIR / "metadata"


def write_manifest(path, fieldnames, rows, key):
    """Write a CSV manifest; without OVERWRITE, keep existing rows and append only new keys."""
    existing = []
    if not OVERWRITE and path.is_file():
        with path.open(newline="", encoding="utf-8") as input_file:
            existing = [{name: row.get(name, "") for name in fieldnames} for row in csv.DictReader(input_file)]
    known = {row[key] for row in existing}
    new_rows = []
    for row in rows:
        if row[key] not in known:
            known.add(row[key])
            new_rows.append(row)
    with path.open("w", newline="", encoding="utf-8") as output:
        writer = csv.DictWriter(output, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(existing + new_rows)
    return len(existing), len(new_rows)


clip_names = [
    line.split("|", maxsplit=1)[-1].strip()
    for line in clip_list.read_text(encoding="utf-8").splitlines()[2:]
    if line.strip()
]
video_kept, video_added = write_manifest(
    DATA_DIR / "video_names.csv",
    ["video_name"],
    [{"video_name": clip_name} for clip_name in clip_names],
    key="video_name",
)

camera_rows = [
    {"metadata_file": file.name, "camera_name": file.stem.split(".")[-1]}
    for file in sorted(metadata_dir.glob("*.krtd"))
]
camera_kept, camera_added = write_manifest(
    DATA_DIR / "camera_names.csv",
    ["metadata_file", "camera_name"],
    camera_rows,
    key="metadata_file",
)

print(
    f"OVERWRITE={OVERWRITE}: video_names.csv {video_kept} kept + {video_added} added; "
    f"camera_names.csv {camera_kept} kept + {camera_added} added."
)

Stage 2: creating raw-data manifests (OVERWRITE=False)
OVERWRITE=False: video_names.csv 769 kept + 0 added; camera_names.csv 29 kept + 0 added.


## Load MEVA ground truth into `bronze.ground_truth`

`bronze.ground_truth` previously held a manually uploaded, partial snapshot: 161 activity
records covering only 10 of the 769 annotated MEVA clips (a leftover `test_10` subset), and
with no `video_id` column, so notebook 8 had to join it to `bronze.videos` on
`(camera_id, capture_start_local)` — fragile, since `camera_id` repeats across capture windows
and dates.

This cell replaces that snapshot with the full, authoritative set: every activity annotation
parsed directly from `data/raw/annotations/kitware/*.activities.yml` (769 clips, ~31k activity
records), independent of which videos have been downloaded or processed yet (a video the
pipeline hasn't reached simply has no matching row in `bronze.videos`; notebook 8 already
reports that gap as `no_video`).

Each row gets a `video_id` computed the same way as `bronze.videos.video_id`
(`sha256(f"video:{video_name}")`), so it can be joined directly to `bronze.videos` and
`bronze.events` without going through `camera_id`/timestamp matching.

The table is fully truncated and reloaded on every run — the source YAML files are static and
parsing all of them takes seconds, so a full resync is simpler and safer than an incremental
diff against a table that used to be hand-curated.

In [ ]:
import hashlib
import re
from datetime import datetime

import psycopg
import yaml
from dotenv import load_dotenv
from psycopg import sql

load_dotenv(PROJECT_ROOT / ".env")
SUPABASE_DB_URL = os.getenv("SUPABASE_DB_URL")
if not SUPABASE_DB_URL:
    raise EnvironmentError(f"Set SUPABASE_DB_URL in {PROJECT_ROOT / '.env'} before running.")

VIDEO_NAME_RE = re.compile(
    r"(?P<date>\d{4}-\d{2}-\d{2})\.(?P<start>\d{2}-\d{2}-\d{2})\."
    r"(?P<end>\d{2}-\d{2}-\d{2})\.(?P<scene>[^.]+)\.(?P<camera>[^.]+)"
)
KITWARE_DIR = DATA_DIR / "annotations" / "kitware"

GROUND_TRUTH_SCHEMA_SQL = """
CREATE SCHEMA IF NOT EXISTS bronze;

CREATE TABLE IF NOT EXISTS bronze.ground_truth (
    id BIGSERIAL PRIMARY KEY,
    ground_truth_id TEXT,
    video_id TEXT,
    video_name TEXT,
    camera_id TEXT,
    capture_start_local TIMESTAMP,
    capture_end_local TIMESTAMP,
    activity_type TEXT NOT NULL,
    start_frame INTEGER NOT NULL,
    end_frame INTEGER NOT NULL,
    n_actors INTEGER,
    src_status TEXT,
    meva_event_id INTEGER,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
-- Tables created before these columns existed get them added here.
ALTER TABLE bronze.ground_truth ADD COLUMN IF NOT EXISTS ground_truth_id TEXT;
ALTER TABLE bronze.ground_truth ADD COLUMN IF NOT EXISTS video_id TEXT;
ALTER TABLE bronze.ground_truth ADD COLUMN IF NOT EXISTS video_name TEXT;
ALTER TABLE bronze.ground_truth ADD COLUMN IF NOT EXISTS capture_end_local TIMESTAMP;
ALTER TABLE bronze.ground_truth ADD COLUMN IF NOT EXISTS meva_event_id INTEGER;

ALTER TABLE bronze.ground_truth DROP CONSTRAINT IF EXISTS ground_truth_ground_truth_id_key;
ALTER TABLE bronze.ground_truth ADD CONSTRAINT ground_truth_ground_truth_id_key UNIQUE (ground_truth_id);
CREATE INDEX IF NOT EXISTS ground_truth_video_id_idx ON bronze.ground_truth(video_id);
CREATE INDEX IF NOT EXISTS ground_truth_camera_capture_idx
    ON bronze.ground_truth(camera_id, capture_start_local);
"""


def stable_id(*parts):
    """Return a globally stable SHA-256 identifier from namespaced values, matching notebook 7."""
    return hashlib.sha256(":".join(map(str, parts)).encode("utf-8")).hexdigest()


def parse_video_name(video_name):
    """Extract the capture interval, scene, and camera ID embedded in a MEVA clip name."""
    match = VIDEO_NAME_RE.fullmatch(video_name)
    if not match:
        return {}
    values = match.groupdict()
    return {
        "scene": values["scene"],
        "camera_id": values["camera"],
        "capture_start_local": datetime.strptime(
            f"{values['date']} {values['start']}", "%Y-%m-%d %H-%M-%S"
        ),
        "capture_end_local": datetime.strptime(
            f"{values['date']} {values['end']}", "%Y-%m-%d %H-%M-%S"
        ),
    }


def ground_truth_rows(activities_path):
    """Parse one *.activities.yml file into bronze.ground_truth rows."""
    video_name = activities_path.name.removesuffix(".activities.yml")
    parsed_name = parse_video_name(video_name)
    video_id = stable_id("video", video_name)
    records = yaml.safe_load(activities_path.read_text(encoding="utf-8")) or []
    rows = []
    for record in records:
        act = record.get("act") if isinstance(record, dict) else None
        if act is None:
            continue  # Non-activity KPF records (e.g. meta headers) are not ground truth.
        act2 = act.get("act2") or {}
        timespans = act.get("timespan") or []
        if not act2 or not timespans:
            continue
        start_frame, end_frame = timespans[0]["tsr0"]
        meva_event_id = act.get("id2")
        rows.append(
            {
                "ground_truth_id": stable_id("ground_truth", video_name, meva_event_id),
                "video_id": video_id,
                "video_name": video_name,
                "camera_id": parsed_name.get("camera_id"),
                "capture_start_local": parsed_name.get("capture_start_local"),
                "capture_end_local": parsed_name.get("capture_end_local"),
                "activity_type": next(iter(act2)),
                "start_frame": start_frame,
                "end_frame": end_frame,
                "n_actors": len(act.get("actors") or []),
                "src_status": act.get("src_status"),
                "meva_event_id": meva_event_id,
            }
        )
    return rows


activities_paths = sorted(KITWARE_DIR.glob("*.activities.yml"))
if not activities_paths:
    raise FileNotFoundError(f"No *.activities.yml files found under {KITWARE_DIR}")

ground_truth = [row for path in activities_paths for row in ground_truth_rows(path)]
print(
    f"Parsed {len(ground_truth):,} MEVA activity records from {len(activities_paths)} clips.",
    flush=True,
)

with psycopg.connect(SUPABASE_DB_URL) as connection:
    with connection.cursor() as cursor:
        cursor.execute(GROUND_TRUTH_SCHEMA_SQL)
        cursor.execute("TRUNCATE TABLE bronze.ground_truth")
        columns = list(ground_truth[0])
        insert_statement = sql.SQL("INSERT INTO bronze.ground_truth ({}) VALUES ({})").format(
            sql.SQL(", ").join(map(sql.Identifier, columns)),
            sql.SQL(", ").join(sql.Placeholder() * len(columns)),
        )
        values = [tuple(row[column] for column in columns) for row in ground_truth]
        cursor.executemany(insert_statement, values)
    connection.commit()

print(f"Reloaded bronze.ground_truth with {len(ground_truth):,} rows.", flush=True)